### Class attributes

In [1]:
class Account:
    bank_name = "SBI"    # class attribute: sits directly in the class body, not inside __init__

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

acc1 = Account("Kavya", 100)
acc2 = Account("Rahul", 500)

print("acc1.bank_name:", acc1.bank_name)
print("acc2.bank_name:", acc2.bank_name)
print("Account.bank_name:", Account.bank_name)

acc1.bank_name: SBI
acc2.bank_name: SBI
Account.bank_name: SBI


#### write it in __init__ with self.x = ... → each instance gets its own separate copy. Write it directly in the class body → every instance shares the exact same one.

### Attribute lookup order

In [2]:
class Account:
    bank_name = "SBI"

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

acc1 = Account("Kavya", 100)

print("acc1.__dict__:", acc1.__dict__)
print("acc1.bank_name:", acc1.bank_name)

acc1.__dict__: {'owner': 'Kavya', 'balance': 100}
acc1.bank_name: SBI


__dict__ — it has owner and balance, but no bank_name at all. That instance genuinely does not have its own copy of bank_name. Yet acc1.bank_name still returns "SBI" without any error.

Here's what Python actually does the moment you write acc1.bank_name:

Check the instance's own __dict__ first. Is there a key called "bank_name" there? In this case: no.
Since it's not on the instance, fall back to the class. Check Account's own attributes. bank_name = "SBI" is sitting right there in the class body — found it, return "SBI".

#### So the rule is: instance checked first, class checked second, as a fallback 

### Concept: Assigning over a class attribute — shadowing, not mutating

In [3]:
acc1 = Account("Kavya", 100)
acc2 = Account("Rahul", 500)

acc1.bank_name = "HDFC"

print("acc1.bank_name:", acc1.bank_name)
print("acc2.bank_name:", acc2.bank_name)
print("Account.bank_name:", Account.bank_name)
print("acc1.__dict__:", acc1.__dict__)

acc1.bank_name: HDFC
acc2.bank_name: SBI
Account.bank_name: SBI
acc1.__dict__: {'owner': 'Kavya', 'balance': 100, 'bank_name': 'HDFC'}


acc1.__dict__ now — it has a brand new key: 'bank_name': 'HDFC'. That line acc1.bank_name = "HDFC" didn't reach into the class and change the shared "SBI" value. It created a new, separate entry directly on acc1's own instance dict.

you can assign any new key you want, on any single instance, at any point — but it only ever affects that one instance. It's exactly the same rule you already know (self.attr = value → new key in that instance's own dict), just applied to a name that was never mentioned in __init__

### The mutable-default bug

In [4]:
class Account:
    transactions = []   # class attribute — a list, defined once for the whole class

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

acc1 = Account("Kavya", 100)
acc2 = Account("Rahul", 500)

acc1.transactions.append("deposit 50")   # looks like it's only touching acc1

print("acc1.transactions:", acc1.transactions)
print("acc2.transactions:", acc2.transactions)
print("Account.transactions:", Account.transactions)
print("acc1.transactions is acc2.transactions:", acc1.transactions is acc2.transactions)

acc1.transactions: ['deposit 50']
acc2.transactions: ['deposit 50']
Account.transactions: ['deposit 50']
acc1.transactions is acc2.transactions: True


### reassigning (self.x = new_value) creates a new instance attribute and is safe. Mutating (.append(), .update(), etc.) on a class attribute that happens to be a list/dict modifies the one shared object everyone points to — and that's silent and dangerous.

### The mutable-default-argument bug in __init__

In [5]:
class Account:
    def __init__(self, owner, balance=0, transactions=[]):
        self.owner = owner
        self.balance = balance
        self.transactions = transactions   # this LOOKS like a normal instance attribute

acc1 = Account("Kavya", 100)
acc2 = Account("Rahul", 500)

acc1.transactions.append("deposit 50")

print("acc1.transactions:", acc1.transactions)
print("acc2.transactions:", acc2.transactions)
print("acc1.transactions is acc2.transactions:", acc1.transactions is acc2.transactions)

acc1.transactions: ['deposit 50']
acc2.transactions: ['deposit 50']
acc1.transactions is acc2.transactions: True


The difference between this and the buggy version: self.transactions = [] written directly in the body runs that [] fresh, every time __init__ executes — once per account. But def __init__(self, transactions=[]) as a default argument only builds that [] one time, ever, no matter how many accounts get created afterward — and every account that doesn't override it shares that single original list.

In [6]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.transactions = []   # created fresh, right here, every single call

### Concept: instance.__dict__ vs ClassName.__dict__

In [7]:
class Account:
    bank_name = "SBI"

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

acc1 = Account("Kavya", 100)

print("acc1.__dict__:", acc1.__dict__)
print("Account.__dict__ keys:", list(Account.__dict__.keys()))

acc1.__dict__: {'owner': 'Kavya', 'balance': 100}
Account.__dict__ keys: ['__module__', '__firstlineno__', 'bank_name', '__init__', 'deposit', '__static_attributes__', '__dict__', '__weakref__', '__doc__']


Two very different dicts, holding two very different kinds of things:

acc1.__dict__ — only owner and balance. Just the instance attributes that got set via self.x = ... for this specific account.
Account.__dict__ — bank_name (the class attribute), __init__ and deposit (the actual function objects, shared by every instance), plus some Python bookkeeping entries (__module__, __doc__, etc.) you can ignore for now.



**1. Instance attributes** — `self.x = value` inside a method writes into that specific instance's own dict. Every instance gets its own separate copy.

**2. Class attributes** — something like `bank_name = "SBI"` written directly in the class body belongs to the class itself, not any instance. Every instance shares that one single value.

**3. Lookup order** — when you write `acc1.bank_name`, Python checks `acc1`'s own dict first; if it's not there, it falls back to the class. That's how `acc1.bank_name` works even though `bank_name` was never in `acc1.__dict__`.

**4. Shadowing** — `acc1.bank_name = "HDFC"` doesn't change the shared class value. It creates a brand new key inside `acc1`'s own dict, which then wins the lookup for `acc1` only. `acc2` and `Account` stay untouched.

**5. Dynamic attributes** — you can assign a completely new attribute to any instance at any time (`acc1.dob = "..."`), even if it was never mentioned in `__init__` or the class body. It only ever affects that one instance.

**6. Mutable class-attribute bug** — if the class attribute is a list/dict (`transactions = []`), calling `.append()` on it through any instance mutates the *one shared object* itself, rather than reassigning — so every instance sees the change, silently.

**7. Mutable default-argument bug** — `def __init__(self, transactions=[])` builds that `[]` exactly once, at definition time. Every call that skips the argument shares that same original list, even though `self.transactions = transactions` looks like a normal safe instance assignment.

**8. Separate dict ≠ separate value** — each instance's dict really is its own separate dict, always. But a dict slot can hold a *reference* to a shared object. If the value placed there was never freshly created per instance, two separate dicts can still point at the identical object.

**9. `instance.__dict__` vs `ClassName.__dict__`** — the instance's dict holds only what `self.x = ...` set. The class's dict holds the class attributes, the method definitions themselves, and some built-in bookkeeping.
